# 3-Fold CV: LoRA + Linear Probe vs Frozen Baseline

3-fold stratified cross-validation of the winning LoRA configuration from the sweep, with the frozen-features linear probe as a within-fold baseline for fair comparison.

**Hyperparameters:** update the `CONFIG` cell below with the winning values from `01_sweep.ipynb`.

**Outputs:**
- `output_cv_lora/per_fold_per_class_results.csv` — granular per-fold per-class metrics.
- `output_cv_lora/per_class_summary_mean_std.csv` — per-class mean ± std across folds.
- `output_cv_lora/per_class_delta.csv` — per-class delta (LoRA - Frozen).
- `output_cv_lora/headline_metrics_mean_std.csv` — aggregate metrics with mean ± std.
- `output_cv_lora/features_fold{0,1,2}.npz` — cached frozen and LoRA features per fold, for follow-up analyses (PCA, neighborhood purity, etc.).

**Feature files contain:**
- `train_features_frozen`, `val_features_frozen` — BioCLIP 2 frozen features.
- `train_features_lora`, `val_features_lora` — LoRA-modified features.
- `train_labels`, `val_labels` — corresponding class labels.
- `train_idx`, `val_idx` — original indices in the full dataset.

The cached features enable downstream PCA analysis without re-running the expensive feature extraction.

## Imports and configuration

In [1]:
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize as sk_normalize
from torch.utils.data import Dataset, DataLoader
import open_clip
from peft import LoraConfig, get_peft_model

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
CONFIG = {
    "model_name": "hf-hub:imageomics/bioclip-2",
    "embedding_dim": 768,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "meta_path": "/workspace/thesis/ultimate_bioclip_top1_rank_order_meta_12062025.csv",
    "image_root": "/workspace/thesis/ALL_copy",
    "output_dir": "/workspace/thesis/output_cv_lora",

    "image_id_col": "image_id",
    "label_col": "label",

    "k_folds": 3,
    "random_state": 0,
    "min_class_size": 50,

    # ============================================
    # WINNING HYPERPARAMETERS FROM SWEEP — values updated according to 01_sweep.ipynb output
    # ============================================
    "lr": 5e-4,
    "r": 16,
    "epochs": 2,
    # ============================================

    "lora_target_modules": ["out_proj", "c_fc", "c_proj"],
    "lora_alpha_factor": 2,
    "lora_dropout": 0.05,

    "batch_size": 64,
    "weight_decay": 1e-4,
    "num_workers": 4,

    "linear_probe_C": 10.0,
    "linear_probe_max_iter": 5000,

    # Feature saving control
    "save_features": True,  # if set to False it skips feature caching
}

CROP_ID_PATTERN = re.compile(r"(\d{14}_crop_[a-zA-Z0-9]+\.jpg)$")
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

print(f"Hyperparameters: lr={CONFIG['lr']}, r={CONFIG['r']}, epochs={CONFIG['epochs']}")
print(f"Output: {CONFIG['output_dir']}")
print(f"Save features: {CONFIG['save_features']}")

Hyperparameters: lr=0.0005, r=16, epochs=2
Output: /workspace/thesis/output_cv_lora
Save features: True


## Helper functions (same as in sweep notebook - 01_sweep.ipynb)

In [3]:
def build_path_index(image_root):
    """Scan local image directory, build {image_id: full_path}."""
    image_root = Path(image_root)
    files = list(image_root.glob("*.jpg"))
    path_index = {}
    for f in files:
        m = CROP_ID_PATTERN.search(f.name)
        if m:
            path_index[m.group(1)] = str(f)
    print(f"Mapped {len(path_index)} image_ids")
    return path_index


class ImageOrderDataset(Dataset):
    """Dataset of (image, label) pairs loaded from disk."""
    def __init__(self, image_paths, labels, preprocess):
        self.image_paths = image_paths
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.preprocess(img)
        return img, self.labels[idx]

In [4]:
def train_lora(model, train_loader, classnames, config, fold_idx):
    """Applyinh LoRA to image encoder and training with classification head."""
    device = config["device"]
    n_classes = len(classnames)

    lora_config = LoraConfig(
        r=config["r"],
        lora_alpha=config["r"] * config["lora_alpha_factor"],
        lora_dropout=config["lora_dropout"],
        target_modules=config["lora_target_modules"],
        bias="none",
    )
    visual = get_peft_model(model.visual, lora_config).to(device)
    visual.print_trainable_parameters()

    head = nn.Linear(config["embedding_dim"], n_classes).to(device)
    trainable = [p for p in visual.parameters() if p.requires_grad] + list(head.parameters())

    optimizer = torch.optim.AdamW(trainable, lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])

    for epoch in range(config["epochs"]):
        visual.train()
        head.train()
        total_loss, n_batches = 0.0, 0
        ep_start = time.time()
        for batch_idx, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            features = F.normalize(visual(imgs), dim=-1)
            logits = head(features)
            loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
            if batch_idx % 100 == 0:
                print(f"    Fold {fold_idx+1} Ep {epoch+1} batch {batch_idx}/{len(train_loader)}: "
                      f"loss={loss.item():.4f}")
        scheduler.step()
        print(f"    Fold {fold_idx+1} Ep {epoch+1}: avg loss={total_loss/n_batches:.4f} "
              f"({(time.time()-ep_start)/60:.1f} min)")
    return visual


def extract_features(encoder_or_model, dataset, config, is_lora=True):
    """Extract L2-normalized image features."""
    device = config["device"]
    encoder = encoder_or_model if is_lora else encoder_or_model.visual
    encoder.eval()
    loader = DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=False,
        num_workers=config["num_workers"], pin_memory=True,
    )
    all_features = []
    with torch.no_grad():
        for imgs, _ in loader:
            features = F.normalize(encoder(imgs.to(device, non_blocking=True)), dim=-1)
            all_features.append(features.cpu().numpy())
    return np.concatenate(all_features, axis=0)


def fit_linear_probe(train_features, train_labels, val_features, config):
    """Fitting multinomial logistic regression."""
    clf = LogisticRegression(
        C=config["linear_probe_C"],
        max_iter=config["linear_probe_max_iter"],
        solver="lbfgs",
        random_state=config["random_state"],
    )
    clf.fit(train_features, train_labels)
    return clf.predict(val_features)

## Loading model and metadata

In [5]:
print("Loading BioCLIP 2...")
model, _, preprocess = open_clip.create_model_and_transforms(CONFIG["model_name"])
model = model.to(CONFIG["device"])

path_index = build_path_index(CONFIG["image_root"])

meta = pd.read_csv(CONFIG["meta_path"])
meta = meta.dropna(subset=[CONFIG["label_col"], CONFIG["image_id_col"]]).reset_index(drop=True)
meta["local_path"] = meta[CONFIG["image_id_col"]].map(path_index)
matched = meta.dropna(subset=["local_path"]).reset_index(drop=True)

y_str = matched[CONFIG["label_col"]].to_numpy()
image_paths = matched["local_path"].to_numpy()
classnames = sorted(np.unique(y_str).tolist())
class_to_idx = {c: i for i, c in enumerate(classnames)}
y = np.array([class_to_idx[s] for s in y_str])

class_counts = pd.Series(y_str).value_counts()
eligible_classes = class_counts[class_counts >= CONFIG["min_class_size"]].index.tolist()
eligible_class_indices = [class_to_idx[c] for c in eligible_classes]
eligible_class_set = set(eligible_classes)

print(f"Matched: {len(matched)} crops")
print(f"Classes ({len(classnames)}): {classnames}")
print(f"Eligible for LoRA training (n>=50): {eligible_classes}")

Loading BioCLIP 2...


Mapped 48095 image_ids
Matched: 39787 crops
Classes (14): ['Araneae', 'Blattodea', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Ixodida', 'Lepidoptera', 'Mecoptera', 'Neuroptera', 'Orthoptera', 'Plecoptera', 'Raphidioptera', 'Trichoptera']
Eligible for LoRA training (n>=50): ['Diptera', 'Hymenoptera', 'Lepidoptera', 'Coleoptera', 'Hemiptera', 'Araneae', 'Neuroptera', 'Trichoptera', 'Plecoptera']


## Setting up 3-fold stratified cross-validation

Classes with fewer than `k_folds` examples are kept in the training set for every fold (they can never appear in a test fold). For 3-fold CV, this affects classes with n < 3 (typically Ixodida n=1 and Raphidioptera n=2).

In [6]:
class_counts_arr = np.bincount(y, minlength=len(classnames))
splittable_mask = class_counts_arr[y] >= CONFIG["k_folds"]
splittable_indices = np.where(splittable_mask)[0]
unsplittable_indices = np.where(~splittable_mask)[0]

print(f"Splittable samples (n>={CONFIG['k_folds']}): {len(splittable_indices)}")
print(f"Unsplittable samples (n<{CONFIG['k_folds']}, kept in train): {len(unsplittable_indices)}")

skf = StratifiedKFold(n_splits=CONFIG["k_folds"], shuffle=True, random_state=CONFIG["random_state"])
raw_splits = list(skf.split(splittable_indices, y[splittable_indices]))

fold_splits = []
for train_rel, val_rel in raw_splits:
    train_idx = np.concatenate([splittable_indices[train_rel], unsplittable_indices])
    val_idx = splittable_indices[val_rel]
    fold_splits.append((train_idx, val_idx))

for i, (tr, va) in enumerate(fold_splits):
    print(f"  Fold {i+1}: train={len(tr)}, val={len(va)}")

Splittable samples (n>=3): 39784
Unsplittable samples (n<3, kept in train): 3
  Fold 1: train=26525, val=13262
  Fold 2: train=26526, val=13261
  Fold 3: train=26526, val=13261


## Running cross-validation

For each fold: computing frozen baseline + LoRA result. Both use the same train/val split for fair within-fold comparison.

Features are saved after each fold to `features_fold{idx}.npz` for downstream analyses (PCA, etc.).

In [7]:
all_results = []
overall_start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"\n{'='*70}\nFOLD {fold_idx+1}/{CONFIG['k_folds']}\n{'='*70}")
    fold_start = time.time()

    train_mask = np.isin(y[train_idx], eligible_class_indices)
    train_idx_eligible = train_idx[train_mask]

    train_paths_eligible = image_paths[train_idx_eligible]
    train_labels_eligible = y[train_idx_eligible]
    train_paths_full = image_paths[train_idx]
    train_labels_full = y[train_idx]
    val_paths = image_paths[val_idx]
    val_labels = y[val_idx]

    train_dataset_eligible = ImageOrderDataset(train_paths_eligible, train_labels_eligible, preprocess)
    train_dataset_full = ImageOrderDataset(train_paths_full, train_labels_full, preprocess)
    val_dataset = ImageOrderDataset(val_paths, val_labels, preprocess)
    train_loader = DataLoader(train_dataset_eligible, batch_size=CONFIG["batch_size"],
                              shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)

    print(f"  Train (LoRA): {len(train_idx_eligible)}, Train (probe): {len(train_idx)}, Val: {len(val_idx)}")

    # ---- Frozen baseline ----
    print(f"\n  --- Frozen baseline ---")
    baseline_start = time.time()
    train_features_frozen = extract_features(model, train_dataset_full, CONFIG, is_lora=False)
    val_features_frozen = extract_features(model, val_dataset, CONFIG, is_lora=False)
    train_features_frozen = sk_normalize(train_features_frozen, norm="l2", axis=1)
    val_features_frozen = sk_normalize(val_features_frozen, norm="l2", axis=1)
    preds_frozen = fit_linear_probe(train_features_frozen, train_labels_full, val_features_frozen, CONFIG)
    print(f"  Baseline time: {timedelta(seconds=int(time.time() - baseline_start))}")

    report_frozen = classification_report(
        val_labels, preds_frozen, labels=list(range(len(classnames))),
        target_names=classnames, output_dict=True, zero_division=0,
    )
    for cls_name in classnames:
        stats = report_frozen.get(cls_name, {})
        all_results.append({
            "method": "frozen_linear_probe", "fold": fold_idx, "class": cls_name,
            "precision": round(stats.get("precision", 0.0), 4),
            "recall": round(stats.get("recall", 0.0), 4),
            "f1": round(stats.get("f1-score", 0.0), 4),
            "support": int(stats.get("support", 0)),
            "trained_in_lora": cls_name in eligible_class_set,
        })

    # ---- LoRA + linear probe ----
    print(f"\n  --- LoRA + linear probe ---")
    lora_start = time.time()
    model_fresh, _, _ = open_clip.create_model_and_transforms(CONFIG["model_name"])
    model_fresh = model_fresh.to(CONFIG["device"])
    visual = train_lora(model_fresh, train_loader, classnames, CONFIG, fold_idx)
    train_features_lora = extract_features(visual, train_dataset_full, CONFIG, is_lora=True)
    val_features_lora = extract_features(visual, val_dataset, CONFIG, is_lora=True)
    train_features_lora = sk_normalize(train_features_lora, norm="l2", axis=1)
    val_features_lora = sk_normalize(val_features_lora, norm="l2", axis=1)
    preds_lora = fit_linear_probe(train_features_lora, train_labels_full, val_features_lora, CONFIG)
    print(f"  LoRA time: {timedelta(seconds=int(time.time() - lora_start))}")

    report_lora = classification_report(
        val_labels, preds_lora, labels=list(range(len(classnames))),
        target_names=classnames, output_dict=True, zero_division=0,
    )
    for cls_name in classnames:
        stats = report_lora.get(cls_name, {})
        all_results.append({
            "method": "lora_linear_probe", "fold": fold_idx, "class": cls_name,
            "precision": round(stats.get("precision", 0.0), 4),
            "recall": round(stats.get("recall", 0.0), 4),
            "f1": round(stats.get("f1-score", 0.0), 4),
            "support": int(stats.get("support", 0)),
            "trained_in_lora": cls_name in eligible_class_set,
        })

    # ---- Saving features for downstream analysis ----
    if CONFIG["save_features"]:
        feat_path = Path(CONFIG["output_dir"]) / f"features_fold{fold_idx}.npz"
        np.savez_compressed(
            feat_path,
            train_features_frozen=train_features_frozen,
            val_features_frozen=val_features_frozen,
            train_features_lora=train_features_lora,
            val_features_lora=val_features_lora,
            train_labels=train_labels_full,
            val_labels=val_labels,
            train_idx=train_idx,
            val_idx=val_idx,
            classnames=np.array(classnames),
            eligible_classes=np.array(eligible_classes),
        )
        size_mb = feat_path.stat().st_size / (1024 * 1024)
        print(f"  Saved features to {feat_path.name} ({size_mb:.1f} MB)")

    # Cleanup
    del visual, model_fresh
    torch.cuda.empty_cache()

    # Saving intermediate metrics
    pd.DataFrame(all_results).to_csv(
        Path(CONFIG["output_dir"]) / "per_fold_per_class_results.csv", index=False
    )
    print(f"  Fold {fold_idx+1} total: {timedelta(seconds=int(time.time() - fold_start))}")

print(f"\n{'='*70}\nCV COMPLETE — Total: {timedelta(seconds=int(time.time() - overall_start))}\n{'='*70}")


FOLD 1/3
  Train (LoRA): 26480, Train (probe): 26525, Val: 13262

  --- Frozen baseline ---
  Baseline time: 0:06:28

  --- LoRA + linear probe ---
trainable params: 4,718,592 || all params: 308,684,800 || trainable%: 1.5286
    Fold 1 Ep 1 batch 0/414: loss=2.6690
    Fold 1 Ep 1 batch 100/414: loss=0.9514
    Fold 1 Ep 1 batch 200/414: loss=0.5253
    Fold 1 Ep 1 batch 300/414: loss=0.3736
    Fold 1 Ep 1 batch 400/414: loss=0.2232
    Fold 1 Ep 1: avg loss=0.7692 (11.4 min)
    Fold 1 Ep 2 batch 0/414: loss=0.1902
    Fold 1 Ep 2 batch 100/414: loss=0.2419
    Fold 1 Ep 2 batch 200/414: loss=0.1203
    Fold 1 Ep 2 batch 300/414: loss=0.1768
    Fold 1 Ep 2 batch 400/414: loss=0.1616
    Fold 1 Ep 2: avg loss=0.1924 (11.4 min)
  LoRA time: 0:31:04
  Saved features to features_fold0.npz (214.4 MB)
  Fold 1 total: 0:37:43

FOLD 2/3
  Train (LoRA): 26479, Train (probe): 26526, Val: 13261

  --- Frozen baseline ---
  Baseline time: 0:07:01

  --- LoRA + linear probe ---
trainable params

## Aggregating results: per-class mean and std across folds

In [8]:
results_df = pd.DataFrame(all_results)

summary_rows = []
for method in ["frozen_linear_probe", "lora_linear_probe"]:
    for cls_name in classnames:
        class_data = results_df[(results_df["method"] == method) & (results_df["class"] == cls_name)]
        summary_rows.append({
            "method": method, "class": cls_name,
            "support_total": int(class_data["support"].sum()),
            "f1_mean": round(class_data["f1"].mean(), 4),
            "f1_std": round(class_data["f1"].std(ddof=1), 4),
            "precision_mean": round(class_data["precision"].mean(), 4),
            "precision_std": round(class_data["precision"].std(ddof=1), 4),
            "recall_mean": round(class_data["recall"].mean(), 4),
            "recall_std": round(class_data["recall"].std(ddof=1), 4),
            "trained_in_lora": cls_name in eligible_class_set,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_summary_mean_std.csv", index=False)
summary_df

,method,class,support_total,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,trained_in_lora
0,frozen_linear_probe,Araneae,564,0.9856,0.0057,0.9982,0.0031,0.9734,0.0141,True
1,frozen_linear_probe,Blattodea,23,0.9045,0.0414,1.0000,0.0000,0.8274,0.0676,False
2,frozen_linear_probe,Coleoptera,1143,0.8223,0.0252,0.8976,0.0008,0.7594,0.0422,True
3,frozen_linear_probe,Diptera,31913,0.9828,0.0007,0.9737,0.0005,0.9921,0.0009,True
4,frozen_linear_probe,Hemiptera,775,0.7379,0.0190,0.8597,0.0289,0.6464,0.0164,True
5,frozen_linear_probe,Hymenoptera,3326,0.9255,0.0029,0.9416,0.0085,0.9101,0.0045,True
6,frozen_linear_probe,Ixodida,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
7,frozen_linear_probe,Lepidoptera,1744,0.9850,0.0036,0.9901,0.0026,0.9799,0.0050,True
8,frozen_linear_probe,Mecoptera,34,0.2715,0.1044,0.7000,0.2646,0.1742,0.0798,False
9,frozen_linear_probe,Neuroptera,115,0.5534,0.0985,0.7903,0.1262,0.4269,0.0852,True


## Headline aggregate metrics

In [9]:
headline_rows = []
for method in ["frozen_linear_probe", "lora_linear_probe"]:
    per_fold_weighted = []
    per_fold_macro_all = []
    per_fold_macro_eligible = []
    per_fold_hemi = []
    per_fold_cole = []

    for fold in range(CONFIG["k_folds"]):
        fold_data = results_df[(results_df["method"] == method) & (results_df["fold"] == fold)]
        per_fold_weighted.append(np.average(fold_data["f1"], weights=fold_data["support"]))
        per_fold_macro_all.append(fold_data["f1"].mean())
        eligible_data = fold_data[fold_data["class"].isin(eligible_class_set)]
        per_fold_macro_eligible.append(eligible_data["f1"].mean())
        per_fold_hemi.append(fold_data[fold_data["class"] == "Hemiptera"]["f1"].iloc[0])
        per_fold_cole.append(fold_data[fold_data["class"] == "Coleoptera"]["f1"].iloc[0])

    headline_rows.append({
        "method": method,
        "weighted_f1_mean": round(np.mean(per_fold_weighted), 4),
        "weighted_f1_std": round(np.std(per_fold_weighted, ddof=1), 4),
        "macro_f1_all_mean": round(np.mean(per_fold_macro_all), 4),
        "macro_f1_all_std": round(np.std(per_fold_macro_all, ddof=1), 4),
        "macro_f1_eligible_mean": round(np.mean(per_fold_macro_eligible), 4),
        "macro_f1_eligible_std": round(np.std(per_fold_macro_eligible, ddof=1), 4),
        "Hemiptera_f1_mean": round(np.mean(per_fold_hemi), 4),
        "Hemiptera_f1_std": round(np.std(per_fold_hemi, ddof=1), 4),
        "Coleoptera_f1_mean": round(np.mean(per_fold_cole), 4),
        "Coleoptera_f1_std": round(np.std(per_fold_cole, ddof=1), 4),
    })

headline_df = pd.DataFrame(headline_rows)
headline_df.to_csv(Path(CONFIG["output_dir"]) / "headline_metrics_mean_std.csv", index=False)
headline_df

,method,weighted_f1_mean,weighted_f1_std,macro_f1_all_mean,macro_f1_all_std,macro_f1_eligible_mean,macro_f1_eligible_std,Hemiptera_f1_mean,Hemiptera_f1_std,Coleoptera_f1_mean,Coleoptera_f1_std
0,frozen_linear_probe,0.9666,0.0013,0.7122,0.0126,0.8660,0.0156,0.7379,0.0190,0.8223,0.0252
1,lora_linear_probe,0.9779,0.0002,0.6846,0.0125,0.8836,0.0056,0.8816,0.0089,0.8885,0.0068


## Per-class delta (LoRA - Frozen)

In [10]:
delta_rows = []
for cls_name in classnames:
    f = summary_df[(summary_df["method"] == "frozen_linear_probe") & (summary_df["class"] == cls_name)]
    l = summary_df[(summary_df["method"] == "lora_linear_probe") & (summary_df["class"] == cls_name)]
    delta_rows.append({
        "class": cls_name,
        "support": int(f["support_total"].iloc[0]),
        "frozen_f1_mean": f["f1_mean"].iloc[0],
        "frozen_f1_std": f["f1_std"].iloc[0],
        "lora_f1_mean": l["f1_mean"].iloc[0],
        "lora_f1_std": l["f1_std"].iloc[0],
        "delta_mean": round(l["f1_mean"].iloc[0] - f["f1_mean"].iloc[0], 4),
        "trained_in_lora": cls_name in eligible_class_set,
    })
delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_delta.csv", index=False)
delta_df

,class,support,frozen_f1_mean,frozen_f1_std,lora_f1_mean,lora_f1_std,delta_mean,trained_in_lora
0,Araneae,564,0.9856,0.0057,0.9884,0.0041,0.0028,True
1,Blattodea,23,0.9045,0.0414,0.7643,0.0953,-0.1402,False
2,Coleoptera,1143,0.8223,0.0252,0.8885,0.0068,0.0662,True
3,Diptera,31913,0.9828,0.0007,0.9897,0.0003,0.0069,True
4,Hemiptera,775,0.7379,0.0190,0.8816,0.0089,0.1437,True
5,Hymenoptera,3326,0.9255,0.0029,0.9431,0.0034,0.0176,True
6,Ixodida,0,0.0000,0.0000,0.0000,0.0000,0.0000,False
7,Lepidoptera,1744,0.9850,0.0036,0.9868,0.0050,0.0018,True
8,Mecoptera,34,0.2715,0.1044,0.0000,0.0000,-0.2715,False
9,Neuroptera,115,0.5534,0.0985,0.5632,0.0757,0.0098,True


## Output files summary

The notebook generates:

| File | Purpose |
|---|---|
| `per_fold_per_class_results.csv` | Granular per-fold per-class precision/recall/F1 |
| `per_class_summary_mean_std.csv` | Per-class aggregates with mean ± std across folds |
| `per_class_delta.csv` | LoRA vs Frozen per-class deltas |
| `headline_metrics_mean_std.csv` | Top-line aggregates (weighted F1, macro F1, etc.) with mean ± std |
| `features_fold0.npz`, `features_fold1.npz`, `features_fold2.npz` | Cached features for downstream analysis |
